# Phase B — Individual Implementation
## Conservative LSTM with JAX

**Format:** individual, at home. Implement, train, evaluate, and reflect.

**Prerequisite:** the plan your group wrote in Phase A (`A1_Phase_A_Group_Exercise.ipynb`).

<a id='0'></a>
## 0. Environment setup

Install dependencies if you don't have them yet.

In [1]:
# Uncomment if needed (e.g. a fresh environment or Colab)
%pip install -q "jax[cpu]" flax optax matplotlib
    
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import matplotlib.pyplot as plt
import time

print("JAX version:", jax.__version__)
print("Devices    :", jax.devices())

Note: you may need to restart the kernel to use updated packages.
JAX version: 0.12.0.1
Devices    : [CpuDevice(id=0)]


<a id='1'></a>
## 1. Introduction to JAX and Flax

JAX is a numerical computing library built around a **functional** programming style. Its syntax is close to NumPy, with three key additions:

| Function | What it does |
|---|---|
| `jax.grad` | Automatic differentiation (exact gradients of any Python function) |
| `jax.jit` | Just-in-time compilation (XLA) → much faster execution |
| `jax.vmap` | Automatic vectorization over a batch axis |

Random numbers use **explicit keys** (`PRNGKey`) instead of a global seed — the same key always gives the same numbers, which makes your code fully reproducible. Neural networks are built with **Flax** (`flax.linen`, imported as `nn`), a layer on top of JAX for trainable models.

Two things to keep in mind:
- JAX functions should be **pure** (no side effects) so that `jit`/`grad` work correctly.
- JAX arrays are **immutable**: you don't modify them in place, you return new ones.

Run the demo below to see all three capabilities in action.

In [2]:
# Mini-demo of JAX's three key capabilities

# (a) Automatic differentiation
f = lambda x: x ** 3 + 2 * x
df = jax.grad(f)                # derivative: 3x^2 + 2
print("f'(2) =", df(2.0), "(expected 14.0)")

# (b) Explicit random keys -> reproducibility
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)
print("Reproducible sample:", jax.random.normal(k1, (3,)))

# (c) JIT + vmap
g = jax.jit(jax.vmap(f))        # applies f to a whole vector, compiled
print("g([1,2,3]) =", g(jnp.array([1., 2., 3.])))

f'(2) = 14.0 (expected 14.0)
Reproducible sample: [ 1.0040143 -0.9063372 -0.7481722]
g([1,2,3]) = [ 3. 12. 33.]


### Flax basics for recurrent models

- A `nn.Module` describes the *computation*, not the state. You create parameters with `model.init(key, dummy_input)` (returns a `params` pytree), then run the network with `model.apply(params, x)` — `params` is always passed explicitly.
- For recurrence, Flax provides `nn.LSTMCell` (one timestep) and `nn.RNN` (unrolls a cell over a whole sequence). You'll use both below.

### Optax for optimization

```python
optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(params)

loss, grads = jax.value_and_grad(loss_fn)(params, batch)
updates, opt_state = optimizer.update(grads, opt_state, params)
params = optax.apply_updates(params, updates)
```

Wrapping this whole step in `@jax.jit` compiles it once and reuses the compiled version on every call — this is what makes JAX training loops fast.

A simple example is often enough to illustrate the Flax programming model before introducing recurrence. The following network learns a basic regression problem using a small multilayer perceptron.

In [3]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax


# ---------------------------------------------------------
# 1. Define a simple neural network
# ---------------------------------------------------------

class SimpleMLP(nn.Module):
    hidden_size: int = 32

    # @nn.compact is a decorator in Flax that tells the framework:
    # “The layers used inside this method should be created automatically 
    # the first time the method is called.”
    # It is the most common way to define Flax models.

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden_size)(x)
        x = nn.relu(x)
        x = nn.Dense(1)(x)
        return x

# ---------------------------------------------------------
# 2. Create a synthetic regression dataset
# ---------------------------------------------------------

key = jax.random.PRNGKey(0)
key, data_key, init_key = jax.random.split(key, 3)
x_train = jax.random.normal(data_key, shape=(128, 4))
y_train = (
    2.0 * x_train[:, 0]
    - 1.5 * x_train[:, 1]
    + 0.5 * x_train[:, 2]
    + 0.2
)
y_train = y_train[:, None]

# ---------------------------------------------------------
# 3. Create the model and initialize its parameters
# ---------------------------------------------------------

model = SimpleMLP(hidden_size=16)
dummy_input = jnp.ones((1, 4))
params = model.init(init_key, dummy_input)

# ---------------------------------------------------------
# 4. Define the loss function
# ---------------------------------------------------------

def loss_fn(params, x, y):
    predictions = model.apply(params, x)
    loss = jnp.mean((predictions - y) ** 2)
    return loss

# ---------------------------------------------------------
# 5. Create the optimizer
# ---------------------------------------------------------

optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(params)

# ---------------------------------------------------------
# 6. Define one training step
# ---------------------------------------------------------

@jax.jit
def train_step(params, opt_state, x, y):
    loss, gradients = jax.value_and_grad(loss_fn)(
        params,
        x,
        y
    )
    updates, opt_state = optimizer.update(
        gradients,
        opt_state,
        params
    )
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ---------------------------------------------------------
# 7. Train the model
# ---------------------------------------------------------

num_epochs = 1000

for epoch in range(num_epochs):
    params, opt_state, loss = train_step(
        params,
        opt_state,
        x_train,
        y_train
    )
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {float(loss):.6f}")

# ---------------------------------------------------------
# 8. Evaluate the trained model
# ---------------------------------------------------------

predictions = model.apply(params, x_train)
final_loss = jnp.mean(
    (predictions - y_train) ** 2
)
print(f"Final loss: {float(final_loss):.6f}")


# ---------------------------------------------------------
# 9. Test the model with new samples
# ---------------------------------------------------------

x_test = jnp.array([
    [1.0, 2.0, 3.0, 4.0],
    [-1.0, 0.5, 2.0, 1.0]
])
predicted_values = model.apply(params, x_test)
true_values = (
    2.0 * x_test[:, 0]
    - 1.5 * x_test[:, 1]
    + 0.5 * x_test[:, 2]
    + 0.2
)

print("\nPredictions:")
print(predicted_values.squeeze())

print("\nTrue values:")
print(true_values)

Epoch    0 | Loss: 5.076825
Epoch  100 | Loss: 2.333333
Epoch  200 | Loss: 0.558173
Epoch  300 | Loss: 0.102863
Epoch  400 | Loss: 0.051410
Epoch  500 | Loss: 0.036595
Epoch  600 | Loss: 0.029150
Epoch  700 | Loss: 0.024714
Epoch  800 | Loss: 0.022067
Epoch  900 | Loss: 0.019799
Final loss: 0.017727

Predictions:
[ 0.6405681 -1.6374569]

True values:
[ 0.7  -1.55]


<a id='2'></a>
## 2. Scientific background (recap)

Many physical systems obey **conservation laws** (mass, energy, charge, momentum, ...). A conventional neural network trained only to minimize prediction error has no notion of these laws.

Phase A (§2) asked you to compare four ways of incorporating the conservation law and to choose one for your group's plan. This notebook has you implement **two** models: the **baseline** (§5, Option 1 — direct prediction, no physical structure) and **your model** (§6 — whichever of Options 2/3/4 your group chose), then compare them. This is one instance of **physics-informed machine learning**: known structure shrinks the space of functions the network can represent, so a well-chosen architecture can be both physically valid *and* need less data to generalize well.

<a id='3'></a>
## 3. The scenario (recap)

| Symbol | Meaning | Observed? |
|---|---|---|
| `m(t)` | Water mass (volume) in the reservoir | yes |
| `r(t)` | Inflow | yes |
| `q(t)` | Controlled outflow | yes |
| `c(t)` | Hidden consumption | **no** |

$$ m(t+1) = m(t) + r(t) - q(t) - c(t) $$

As in Phase A §1, `c(t)` depends on temperature, hour of day, day of year, and its own small random noise; it is the only unobserved term in the mass balance.

Models will only ever see `x(t) = [m(t), r(t), q(t), temp(t), hour(t), day_year(t)]` as input (matching the variables Phase A identified as driving `c(t)`). The target for training is `m(t+1)`.

<a id='4'></a>
## 4. Task 5.1 — Generate synthetic sequential data

Simulate `N` independent sequences of length `T` (e.g. `N=400`, `T=480` hourly steps ≈ 20 days).

**Design notes / hints:**
- **Inflow / outflow.** The simplest valid design samples `r(t)` and `q(t)` i.i.d. from a fixed range at every timestep (e.g. `r(t) ~ U[2.0, 3.5]`, `q(t) ~ U[1.0, 2.0]`) — they don't need to be autocorrelated or seasonal, since they're always fully observed anyway.
- **Temperature.** Give it a daily cycle plus per-sequence and per-step noise, e.g.
  `temp(t) = 20 + 8·sin(2π(hour(t) − 6)/24) + offset_seq + noise(t)`, where `offset_seq` is drawn once per sequence and `noise(t)` independently at every step.
- **Day of year.** A sequence spans a small window of days (e.g. `T=480` hours ≈ 20 days), so `day_year(t)` is close to constant within a sequence but differs across sequences — draw one starting day per sequence (e.g. `U[0, 365]`) and increment it as `t` advances. Normalize to `[0, 1]` for the network input, same as `hour_norm`.
- **Hidden consumption `c(t)`.** Build it from a base level, a temperature term, and a daily usage profile, then clip to non-negative:

  ```
  hour_profile(t) = 0.5 · (1 + sin(2π(hour(t) − 8)/24))   # peaks around midday/evening
  c_raw(t)        = c0 + α·(temp(t) − mean(temp)) + β·hour_profile(t) + noise_c(t)
  c(t)            = max(0, c_raw(t))                       # consumption can't be negative
  ```

  (e.g. `c0=1.0`, `α=0.05`, `β=1.2`, small-amplitude `noise_c`). Note it's the **clipping**, not the formula itself, that enforces non-negativity. (You could also make `c0`/`α`/`β` depend on `day_year(t)` — e.g. a seasonal component — but a constant base level is enough to complete the task.)
- Roll the mass forward: pick `m(0)` (e.g. `100 + U[0,10]`), then apply `m(t+1) = m(t) + r(t) - q(t) - c(t)` step by step to build the whole sequence.
- Store `X = [m, r, q, temp, hour_norm, day_year_norm]` (that feature order), `Y = m(t+1)`, and `C` (true hidden consumption, diagnostics only).

In [18]:
def generate_data(key, n_seq=400, T=480):
    """Generate n_seq sequences of length T for the water-reservoir system.

    Returns
    -------
    X : (n_seq, T, 6) array -> [m, r, q, temp, hour_norm, day_year_norm]
    Y : (n_seq, T)    array -> m(t+1)
    C : (n_seq, T)    array -> true hidden consumption (diagnostics only)
    """
    # TODO: implement the generator following the design notes above.
    raise NotImplementedError

key = jax.random.PRNGKey(42)
key, kdata = jax.random.split(key)
# X, Y, C = generate_data(kdata)
# print("X:", X.shape, " Y:", Y.shape, " C:", C.shape)


In [5]:
# TODO: sanity-check plot — visualize one example sequence
# (e.g. mass evolution m(t+1), and the flows r, q, c on a second axis)


**Split and normalize.** Split train/test **by sequence**, not by timestep, so no timesteps from a test sequence leak into training through shared history. Normalize inputs (zero mean, unit variance) using statistics computed **only on the training split**. Keep the *unnormalized* `m(t)`, `r(t)`, `q(t)` around separately — you'll need the raw physical values later for the conservation-law reconstruction, which must not be done in normalized units.

In [6]:
n_train = 300

# TODO: split X, Y, C into *_tr / *_te by sequence using n_train
# X_tr, X_te = ...
# Y_tr, Y_te = ...
# C_tr, C_te = ...

# TODO: compute x_mean, x_std from X_tr ONLY, then define norm_x(x)
# x_mean = ...
# x_std  = ...
# def norm_x(x):
#     return (x - x_mean) / x_std


<a id='5'></a>
## 5. Task 5.2 — Conventional LSTM (the baseline, Option 1)

A standard LSTM that receives the observable variables and **predicts the mass directly**:

$$ \hat{m}(t+1) = \text{LSTM}_\theta\big(x(t)\big) $$

It has no knowledge of the conservation law — it has to learn everything from data. This is your baseline.

In [7]:
class LSTMBackbone(nn.Module):
    """Recurrent LSTM layer returning the hidden state at every step."""
    hidden_size: int = 32

    @nn.compact
    def __call__(self, xs):                     # xs: (batch, T, features)
        # TODO: build an nn.LSTMCell wrapped in nn.RNN and return its output
        raise NotImplementedError


class ConventionalLSTM(nn.Module):
    """Predicts m(t+1) directly."""
    hidden_size: int = 32

    @nn.compact
    def __call__(self, xs):                     # xs (normalized): (batch, T, 6)
        # TODO: LSTMBackbone -> Dense(1) -> squeeze last dim -> (batch, T)
        raise NotImplementedError


<a id='6'></a>
## 6. Task 5.3 — Your model (the option your group chose in Phase A)

Implement **one** of the three non-baseline options from Phase A §2 — whichever your group chose and justified in Discussion 3. You only need the one matching your choice; the other two stubs below are there for reference (or if you want to compare more than one — optional, not required).

**Important, for all three options:** the model should receive `xs` **normalized** for the neural-network part, but must use the **original physical values** of `m`, `r`, `q` wherever the mass balance is used (reconstruction, or the physics loss term) — doing that arithmetic in normalized units would be physically meaningless.

### Option 2 — Soft constraint

Two outputs, `(m̂(t+1), ĉ(t))`, both learned — nothing in the architecture enforces the conservation law. Instead, an extra loss term penalizes violations:

$$ L = L_{\text{pred}} + \lambda\, L_{\text{physics}}, \qquad
L_{\text{pred}} = \big(\hat m(t{+}1)-m(t{+}1)\big)^2, \qquad
L_{\text{physics}} = \big(\hat m(t{+}1)-m(t)-r(t)+q(t)+\hat c(t)\big)^2 $$

`λ` is a new hyperparameter you choose — see §7 for how it plugs into training.

In [8]:
class SoftConstraintLSTM(nn.Module):
    """Option 2: predicts (m_hat, c_hat); conservation is only encouraged via the loss (see Sec.7)."""
    hidden_size: int = 32

    @nn.compact
    def __call__(self, xs_norm):
        # xs_norm: normalized network input (batch, T, 6)
        # TODO:
        #   1. h = LSTMBackbone(...)(xs_norm)
        #   2. two separate Dense(1) heads on h -> m_hat, c_hat, each squeezed to (batch, T)
        #   3. return m_hat, c_hat
        raise NotImplementedError


### Option 3 — Hard constraint (conservative architecture)

The network predicts only the hidden consumption; the mass is reconstructed analytically, so conservation holds exactly by construction:

$$ \hat{c}(t) = \text{softplus}\big(\text{LSTM}_\theta(x(t))\big) \ge 0
\qquad\Longrightarrow\qquad
\hat{m}(t+1) = m(t) + r(t) - q(t) - \hat{c}(t) $$

`softplus` is preferred over `relu` here: it's smooth and differentiable everywhere, so it won't stall learning at `ĉ=0`.

In [9]:
class ConservativeLSTM(nn.Module):
    """Option 3: predicts hidden consumption c_hat(t) >= 0 and closes the mass balance exactly."""
    hidden_size: int = 32

    @nn.compact
    def __call__(self, xs_norm, m_phys, r_phys, q_phys):
        # xs_norm: normalized network input          (batch, T, 6)
        # m_phys, r_phys, q_phys: RAW physical values (batch, T)
        # TODO:
        #   1. h = LSTMBackbone(...)(xs_norm)
        #   2. c_raw = Dense(1) head on h, squeeze to (batch, T)
        #   3. c_hat = nn.softplus(c_raw)              # hard positivity constraint
        #   4. m_pred = m_phys + r_phys - q_phys - c_hat
        #   5. return m_pred, c_hat
        raise NotImplementedError


### Option 4 — Residual correction

Like Option 3, but the network also predicts a small correction `δ(t)` to absorb unmodelled effects:

$$ \hat{m}(t+1) = m(t) + r(t) - q(t) - \hat{c}(t) + \delta(t) $$

Useful if you think the conservation law alone doesn't fully capture the true dynamics — but conservation is no longer exact, so watch how large `δ(t)` grows in §8's physical-error metric.

In [10]:
class ResidualCorrectionLSTM(nn.Module):
    """Option 4: hard constraint plus a small learned correction delta(t)."""
    hidden_size: int = 32

    @nn.compact
    def __call__(self, xs_norm, m_phys, r_phys, q_phys):
        # xs_norm: normalized network input          (batch, T, 6)
        # m_phys, r_phys, q_phys: RAW physical values (batch, T)
        # TODO:
        #   1. h = LSTMBackbone(...)(xs_norm)
        #   2. c_raw = Dense(1) head on h, squeeze to (batch, T); c_hat = nn.softplus(c_raw)
        #   3. delta = a second Dense(1) head on h, squeeze to (batch, T)  # small correction, no positivity constraint
        #   4. m_pred = m_phys + r_phys - q_phys - c_hat + delta
        #   5. return m_pred, c_hat, delta
        raise NotImplementedError


<a id='7'></a>
## 7. Task 5.4 — Train the baseline model and your model under identical conditions

- **Baseline (§5, Option 1):** MSE on `m(t+1)` directly.
- **Your model (§6):** the loss depends on which option you implemented —
  - **Option 2 (soft constraint):** `L = L_pred + λ·L_physics` (§6). `λ` is an extra hyperparameter — try a few values (e.g. `0.1, 1.0, 10.0`) and see how it trades off prediction accuracy against conservation.
  - **Option 3 (hard constraint) / Option 4 (residual correction):** MSE on the *reconstructed* `m̂(t+1)`, not on `ĉ(t)` directly — the true `c(t)` must not be used for supervision.
- Same optimizer, learning rate, hidden size, and number of epochs for both models — otherwise you're no longer measuring the effect of the architecture alone. Reasonable starting values: `hidden_size=32`, `Adam(learning_rate=1e-2)`, `~300` epochs, `batch_size=32` sequences.
- **Use mini-batch training**: at each step, sample (or shuffle-and-slice) a batch of `batch_size` sequences from the training set rather than using all sequences at once. Reshuffle sequence order at the start of every epoch, using `jax.random.permutation` (splitting your key each time) so the shuffling stays reproducible.
- Wrap the training step in `jax.jit` and record training time for both models.
- Plot training loss curves for both on the same axes (log scale is usually clearer).

In [11]:
def make_train_step(loss_fn, optimizer):
    @jax.jit
    def step(params, opt_state, batch):
        loss, grads = jax.value_and_grad(loss_fn)(params, batch)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss
    return step

HIDDEN_SIZE = 32
EPOCHS = 300
BATCH_SIZE = 32
optimizer = optax.adam(1e-2)

# TODO: write a mini-batch training loop you can reuse for both models:
#   for each epoch:
#     - shuffle sequence order (jax.random.permutation, split your key)
#     - slice into batches of BATCH_SIZE
#     - call the jitted step on each batch, accumulate/record the loss
def train_model(model, params, loss_fn, key, X_xs, extra_arrays, y, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """extra_arrays: dict of any additional per-batch arrays the loss_fn needs
    (e.g. {'m': m_phys, 'r': r_phys, 'q': q_phys} for the conservative model)."""
    raise NotImplementedError

In [12]:
# --- BASELINE model (Option 1, Sec.5) ---
# TODO:
#   conv_model = ConventionalLSTM(hidden_size=HIDDEN_SIZE)
#   key, k = jax.random.split(key)
#   conv_params = conv_model.init(k, Xn_tr[:1])
#   define conv_loss(params, batch) -> MSE
#   train it with train_model / your training loop, timing it with time.time()


In [13]:
# --- YOUR MODEL (Sec.6 -- Option 2, 3, or 4, whichever you implemented) ---
# TODO:
#   your_model = <YourChosenClass>(hidden_size=HIDDEN_SIZE)
#   key, k = jax.random.split(key)
#   your_params = your_model.init(k, ...)   # args depend on which option you chose:
#       Option 2:    your_model.init(k, Xn_tr[:1])
#       Option 3/4:  your_model.init(k, Xn_tr[:1], X_tr[:1,:,0], X_tr[:1,:,1], X_tr[:1,:,2])
#   define your_loss(params, batch) -> see the loss recipe for your option in Sec.7
#   train it the same way, timing it


In [14]:
# TODO: plot both training-loss curves on the same (log-scale) axes


<a id='8'></a>
## 8. Metrics and comparison

Compute all four metrics on the **held-out test set** for both models:

| Metric | Formula |
|---|---|
| **MSE** | `mean((m̂(t+1) - m(t+1))^2)` |
| **MAE** | `mean(\|m̂(t+1) - m(t+1)\|)` |
| **Physical error** | `mean(max(0, -(m(t) + r(t) - q(t) - m̂(t+1))))` — magnitude of implied negative consumption |
| **Training time** | wall-clock time for the training loop |

**Expected pattern (depends on which option you implemented in §6):**
- The **baseline** (Option 1) should show a non-zero physical error — nothing stops it from implying negative consumption.
- **Option 2 (soft constraint):** physical error should be *reduced* relative to the baseline but generally still nonzero — how much depends on `λ`; larger `λ` should push it closer to zero, at some cost to MSE.
- **Option 3 (hard constraint):** physical error should be **exactly zero**, always — not just small. `softplus` guarantees `ĉ(t) ≥ 0`, so the implied consumption `m(t)+r(t)-q(t)-m̂(t+1) = ĉ(t)` can never be negative, regardless of training quality.
- **Option 4 (residual correction):** unlike Option 3, `δ(t)` has no positivity constraint, so physical error is generally nonzero — if `δ(t)` grows large, it can exceed Option 3's.

If your results don't roughly match the pattern for your chosen option, check: are both models trained under identical conditions? Does your model's mass computation (or physics loss term) use raw, unnormalized `m, r, q`? Is `c(t)` leaking into either model's inputs anywhere?

In [15]:
def compute_metrics(m_pred, m, r, q, y):
    """Return (mse, mae, physical_error) — see formulas above."""
    # TODO
    raise NotImplementedError

# TODO: run both models on the test set, compute metrics, and print a comparison table
# (Metric | Baseline | Your model), plus training time for each.


In [16]:
# TODO: visual comparison plot(s) on one test sequence:
#   - m(t+1) real vs. predicted by both models
#   - (your model only, if it predicts c_hat(t)) c_hat(t) vs. the true hidden consumption C_te


<a id='10'></a>
## 9. Reflection questions

Answer using evidence from **your own** experiments — refer to your metrics table and plots, not just general reasoning.

**1. Why can a conventional LSTM violate a physical law, even after being trained to convergence on plenty of data?**

_Your answer:_

*(write here)*

**2. What is the difference between a soft constraint and a hard constraint? If you implemented Option 3 or 4, where would a soft constraint have gone instead, and how would training/evaluation have differed? If you implemented Option 2, reflect on how your choice of `λ` affected the accuracy/conservation trade-off.**

_Your answer:_

*(write here)*

**3. How does physical knowledge act as an inductive bias? Explain how you would test this claim with the optional low-data ablation described in the assignment, and connect it to your own results if you ran it.**

_Your answer:_

*(write here)*

**4. In what other application domains could this idea (predicting an unknown residual term and reconstructing the target analytically) be useful? Give at least one concrete example outside of hydrology.**

_Your answer:_

*(write here)*

**5. How could causality, rather than mass conservation, be incorporated into a neural network architecture? What would "hard enforcement" of a causal constraint even mean, compared to enforcing a conservation law?**

_Your answer:_

*(write here)*

<a id='11'></a>
## 11. Before you submit

**Common pitfalls to double-check:**
- The true `c(t)` never leaks into either model's inputs.
- Train/test split is by *sequence*, not by timestep.
- Normalization statistics come from the training split only.
- Your model's mass computation (Options 3/4) or physics loss term (Option 2) uses raw, unnormalized `m, r, q`.
- Both models used the same hidden size, optimizer, learning rate, epochs, and batch size.
- You didn't reuse a JAX PRNG key without splitting it first.

**Deliverables checklist:**
- [ ] Reproducible notebook, seeded end-to-end with explicit `PRNGKey`s
- [ ] Synthetic data generator with a sanity-check plot of at least one simulated sequence
- [ ] Baseline LSTM implementation (Option 1)
- [ ] Your chosen model implementation (Option 2, 3, or 4 -- matching your Phase A plan)
- [ ] Both models trained under identical settings, with training-loss curves plotted together
- [ ] Metrics table (MSE, MAE, physical error, training time) on the test set for both models
- [ ] *(recommended, optional bonus)* Low-data ablation plot (test MSE vs. training-set size)
- [ ] Written answers to all five reflection questions, grounded in your own results